# CRAM File Parser (v3.0)

A standalone, pure-Python parser for the CRAM compressed alignment format.

CRAM is a reference-based compressed alternative to BAM for storing aligned sequencing reads.
It uses multiple codecs (gzip, bzip2, lzma, rANS) and bit-level encoding schemes
(Huffman, beta, gamma, etc.) to achieve high compression ratios.

**Scope:** CRAM v3.0 decoding using only Python standard library modules.

In [ ]:
# Section 0: Imports & Exception Hierarchy
import struct
import zlib
import bz2
import lzma
import io
import os
import hashlib
from collections import OrderedDict
from typing import Dict, List, Optional, Any, Tuple, BinaryIO


class CRAMError(Exception):
    """Base exception for CRAM parsing errors."""
    pass

class CRAMVersionError(CRAMError):
    """Raised when an unsupported CRAM version is encountered."""
    pass

class CRAMDecompressionError(CRAMError):
    """Raised when block decompression fails."""
    pass

class CRAMDecodingError(CRAMError):
    """Raised when data decoding fails."""
    pass

print("CRAM parser modules loaded.")

## Section 1: ITF-8 & LTF-8 Integer Encoding

CRAM uses two variable-length integer formats throughout:

- **ITF-8** (Integer Transformation Format, 8-bit): Encodes 32-bit signed integers in 1-5 bytes.
  The number of leading 1-bits in the first byte indicates total byte count.
- **LTF-8** (Long Transformation Format, 8-bit): Encodes 64-bit integers in 1-9 bytes.

| First byte pattern | Total bytes | Value bits |
|---|---|---|
| `0xxxxxxx` | 1 | 7 |
| `10xxxxxx` | 2 | 14 |
| `110xxxxx` | 3 | 21 |
| `1110xxxx` | 4 | 28 |
| `11110xxx` | 5 | 32 |

In [ ]:
# Section 1: ITF-8 and LTF-8 decoders

def read_itf8(stream: BinaryIO) -> int:
    """Read an ITF-8 encoded 32-bit integer from a binary stream."""
    b = stream.read(1)
    if not b:
        raise CRAMDecodingError("Unexpected end of stream reading ITF-8")
    first = b[0]

    if first < 0x80:  # 0xxxxxxx
        return first
    elif first < 0xC0:  # 10xxxxxx
        b2 = stream.read(1)
        if not b2:
            raise CRAMDecodingError("Unexpected end of stream reading ITF-8")
        return ((first & 0x3F) << 8) | b2[0]
    elif first < 0xE0:  # 110xxxxx
        data = stream.read(2)
        if len(data) < 2:
            raise CRAMDecodingError("Unexpected end of stream reading ITF-8")
        return ((first & 0x1F) << 16) | (data[0] << 8) | data[1]
    elif first < 0xF0:  # 1110xxxx
        data = stream.read(3)
        if len(data) < 3:
            raise CRAMDecodingError("Unexpected end of stream reading ITF-8")
        return ((first & 0x0F) << 24) | (data[0] << 16) | (data[1] << 8) | data[2]
    else:  # 11110xxx
        data = stream.read(4)
        if len(data) < 4:
            raise CRAMDecodingError("Unexpected end of stream reading ITF-8")
        return ((first & 0x07) << 28) | (data[0] << 20) | (data[1] << 12) | (data[2] << 4) | (data[3] & 0x0F)


def read_ltf8(stream: BinaryIO) -> int:
    """Read an LTF-8 encoded 64-bit integer from a binary stream."""
    b = stream.read(1)
    if not b:
        raise CRAMDecodingError("Unexpected end of stream reading LTF-8")
    first = b[0]

    if first < 0x80:
        return first
    elif first < 0xC0:
        data = stream.read(1)
        return ((first & 0x3F) << 8) | data[0]
    elif first < 0xE0:
        data = stream.read(2)
        return ((first & 0x1F) << 16) | (data[0] << 8) | data[1]
    elif first < 0xF0:
        data = stream.read(3)
        return ((first & 0x0F) << 24) | (data[0] << 16) | (data[1] << 8) | data[2]
    elif first < 0xF8:
        data = stream.read(4)
        return ((first & 0x07) << 28) | (data[0] << 20) | (data[1] << 12) | (data[2] << 4) | (data[3] & 0x0F)
    elif first < 0xFC:
        data = stream.read(5)
        return ((first & 0x03) << 36) | (data[0] << 28) | (data[1] << 20) | (data[2] << 12) | (data[3] << 4) | (data[4] & 0x0F)
    elif first < 0xFE:
        data = stream.read(6)
        return ((first & 0x01) << 44) | (data[0] << 36) | (data[1] << 28) | (data[2] << 20) | (data[3] << 12) | (data[4] << 4) | (data[5] & 0x0F)
    elif first == 0xFE:
        data = stream.read(7)
        return (data[0] << 44) | (data[1] << 36) | (data[2] << 28) | (data[3] << 20) | (data[4] << 12) | (data[5] << 4) | (data[6] & 0x0F)
    else:  # 0xFF
        data = stream.read(8)
        return struct.unpack('>q', data)[0]


print("ITF-8 / LTF-8 decoders defined.")

In [ ]:
# Section 1: ITF-8 / LTF-8 unit tests

def test_itf8():
    # 1-byte: value 0
    assert read_itf8(io.BytesIO(b'\x00')) == 0
    # 1-byte: value 127
    assert read_itf8(io.BytesIO(b'\x7f')) == 127
    # 2-byte: value 128 -> 0x80 | 0x00, 0x80
    assert read_itf8(io.BytesIO(b'\x80\x80')) == 128
    # 2-byte: value 16383 -> 0xBF, 0xFF
    assert read_itf8(io.BytesIO(b'\xbf\xff')) == 16383
    # 1-byte: value 5
    assert read_itf8(io.BytesIO(b'\x05')) == 5
    print("ITF-8 tests passed.")

def test_ltf8():
    assert read_ltf8(io.BytesIO(b'\x00')) == 0
    assert read_ltf8(io.BytesIO(b'\x7f')) == 127
    assert read_ltf8(io.BytesIO(b'\x80\x80')) == 128
    print("LTF-8 tests passed.")

test_itf8()
test_ltf8()

## Section 2: Block Decompression

CRAM blocks can be compressed with different methods:

| Method ID | Codec |
|---|---|
| 0 | Raw (no compression) |
| 1 | Gzip (zlib) |
| 2 | Bzip2 |
| 3 | LZMA |
| 4 | rANS (Asymmetric Numeral Systems) |

rANS is a fast entropy coder used in CRAM v3.0. We implement order-0 and order-1 decoders.

In [ ]:
# Section 2: Decompression dispatch + rANS decoder

def _rans_decode_order0(data: bytes) -> bytes:
    """Pure-Python rANS order-0 decoder."""
    src = io.BytesIO(data)

    # Read frequency table
    freqs = [0] * 256
    cum_freqs = [0] * 257
    
    # Read the frequency table
    sym = src.read(1)[0]
    last_sym = sym
    rle = 0
    
    while True:
        f_byte = src.read(1)[0]
        if f_byte & 0x80:
            f = ((f_byte & 0x7F) << 8) | src.read(1)[0]
        else:
            f = f_byte
        freqs[sym] = f
        
        if rle > 0:
            rle -= 1
            sym += 1
        else:
            sym = src.read(1)[0]
            if sym == 0:
                break
            if sym == last_sym + 1:
                rle = src.read(1)[0]
            last_sym = sym

    # Build cumulative frequency table
    TOTFREQ = 1 << 12  # 4096
    cum_freqs[0] = 0
    for i in range(256):
        cum_freqs[i + 1] = cum_freqs[i] + freqs[i]

    # Build lookup table for fast decoding
    lookup = [0] * TOTFREQ
    for sym in range(256):
        for j in range(freqs[sym]):
            lookup[cum_freqs[sym] + j] = sym

    # Read output size
    out_size = struct.unpack('<I', src.read(4))[0]

    # Read initial rANS states (4 interleaved streams)
    R = [0] * 4
    for i in range(4):
        R[i] = struct.unpack('<I', src.read(4))[0]

    compressed = src.read()
    cp = 0
    output = bytearray(out_size)

    mask = TOTFREQ - 1
    for i in range(out_size):
        idx = i & 3
        r = R[idx]
        m = r & mask
        s = lookup[m]
        output[i] = s
        r = freqs[s] * (r >> 12) + m - cum_freqs[s]
        # Renormalize
        while r < (1 << 23):
            if cp < len(compressed):
                r = (r << 8) | compressed[cp]
                cp += 1
            else:
                r = (r << 8)
        R[idx] = r

    return bytes(output)


def _rans_decode_order1(data: bytes) -> bytes:
    """Pure-Python rANS order-1 decoder."""
    src = io.BytesIO(data)
    TOTFREQ = 1 << 12

    # Read frequency table: context -> symbol -> frequency
    freqs = [[0] * 256 for _ in range(256)]
    cum_freqs = [[0] * 257 for _ in range(256)]
    lookup = [[0] * TOTFREQ for _ in range(256)]

    ctx = src.read(1)[0]
    last_ctx = ctx
    ctx_rle = 0

    while True:
        sym = src.read(1)[0]
        last_sym = sym
        sym_rle = 0

        while True:
            f_byte = src.read(1)[0]
            if f_byte & 0x80:
                f = ((f_byte & 0x7F) << 8) | src.read(1)[0]
            else:
                f = f_byte
            freqs[ctx][sym] = f

            if sym_rle > 0:
                sym_rle -= 1
                sym += 1
            else:
                sym = src.read(1)[0]
                if sym == 0:
                    break
                if sym == last_sym + 1:
                    sym_rle = src.read(1)[0]
                last_sym = sym

        # Build cumulative freq for this context
        cum_freqs[ctx][0] = 0
        for i in range(256):
            cum_freqs[ctx][i + 1] = cum_freqs[ctx][i] + freqs[ctx][i]
        for s in range(256):
            for j in range(freqs[ctx][s]):
                lookup[ctx][cum_freqs[ctx][s] + j] = s

        if ctx_rle > 0:
            ctx_rle -= 1
            ctx += 1
        else:
            ctx = src.read(1)[0]
            if ctx == 0:
                break
            if ctx == last_ctx + 1:
                ctx_rle = src.read(1)[0]
            last_ctx = ctx

    out_size = struct.unpack('<I', src.read(4))[0]

    R = [0] * 4
    for i in range(4):
        R[i] = struct.unpack('<I', src.read(4))[0]

    compressed = src.read()
    cp = 0
    output = bytearray(out_size)

    last = [0, 0, 0, 0]
    mask = TOTFREQ - 1

    for i in range(out_size):
        idx = i & 3
        r = R[idx]
        c = last[idx]
        m = r & mask
        s = lookup[c][m]
        output[i] = s
        r = freqs[c][s] * (r >> 12) + m - cum_freqs[c][s]
        while r < (1 << 23):
            if cp < len(compressed):
                r = (r << 8) | compressed[cp]
                cp += 1
            else:
                r = (r << 8)
        R[idx] = r
        last[idx] = s

    return bytes(output)


def decompress_block(method: int, compressed_data: bytes, uncompressed_size: int) -> bytes:
    """Decompress a CRAM block given its compression method."""
    if method == 0:  # Raw
        return compressed_data
    elif method == 1:  # Gzip
        try:
            return zlib.decompress(compressed_data, 15 + 32)  # auto-detect gzip/zlib
        except zlib.error:
            try:
                return zlib.decompress(compressed_data)
            except zlib.error as e:
                raise CRAMDecompressionError(f"Gzip decompression failed: {e}")
    elif method == 2:  # Bzip2
        try:
            return bz2.decompress(compressed_data)
        except Exception as e:
            raise CRAMDecompressionError(f"Bzip2 decompression failed: {e}")
    elif method == 3:  # LZMA
        try:
            return lzma.decompress(compressed_data)
        except Exception as e:
            raise CRAMDecompressionError(f"LZMA decompression failed: {e}")
    elif method == 4:  # rANS
        try:
            if len(compressed_data) < 1:
                raise CRAMDecompressionError("rANS data too short")
            order = compressed_data[0]
            payload = compressed_data[1:]
            if order == 0:
                return _rans_decode_order0(payload)
            elif order == 1:
                return _rans_decode_order1(payload)
            else:
                raise CRAMDecompressionError(f"Unknown rANS order: {order}")
        except CRAMDecompressionError:
            raise
        except Exception as e:
            raise CRAMDecompressionError(f"rANS decompression failed: {e}")
    else:
        raise CRAMDecompressionError(f"Unknown compression method: {method}")


print("Decompression functions defined.")

In [ ]:
# Section 2: Decompression tests

def test_decompression():
    test_data = b"Hello, CRAM parser! This is a test of compression."

    # Raw
    assert decompress_block(0, test_data, len(test_data)) == test_data

    # Gzip
    gz = zlib.compress(test_data)
    assert decompress_block(1, gz, len(test_data)) == test_data

    # Bzip2
    bz = bz2.compress(test_data)
    assert decompress_block(2, bz, len(test_data)) == test_data

    # LZMA
    lz = lzma.compress(test_data)
    assert decompress_block(3, lz, len(test_data)) == test_data

    print("Decompression tests passed (raw, gzip, bzip2, lzma).")
    print("Note: rANS tested implicitly during real CRAM file parsing.")

test_decompression()

## Section 3: Bit-Level I/O

Several CRAM encoding schemes (Huffman, beta, gamma, subexponential) operate at the bit level.
The `BitReader` class provides MSB-first bit reading from a byte buffer.

In [ ]:
# Section 3: BitReader

class BitReader:
    """Reads individual bits from a byte buffer, MSB first."""

    def __init__(self, data: bytes):
        self.data = data
        self.byte_pos = 0
        self.bit_pos = 7  # MSB first

    def read_bit(self) -> int:
        if self.byte_pos >= len(self.data):
            raise CRAMDecodingError("BitReader: no more data")
        bit = (self.data[self.byte_pos] >> self.bit_pos) & 1
        self.bit_pos -= 1
        if self.bit_pos < 0:
            self.bit_pos = 7
            self.byte_pos += 1
        return bit

    def read_bits(self, n: int) -> int:
        val = 0
        for _ in range(n):
            val = (val << 1) | self.read_bit()
        return val

    def read_bytes(self, n: int) -> bytes:
        result = bytearray()
        for _ in range(n):
            result.append(self.read_bits(8))
        return bytes(result)


# Quick test
br = BitReader(b'\xA5')  # 10100101
bits = [br.read_bit() for _ in range(8)]
assert bits == [1, 0, 1, 0, 0, 1, 0, 1], f"Got {bits}"

br2 = BitReader(b'\xFF\x00')
assert br2.read_bits(8) == 0xFF
assert br2.read_bits(8) == 0x00

print("BitReader tests passed.")

## Section 4: Encoding Schemes

CRAM uses multiple encoding types to store data series values:

| Code | Encoding | Description |
|---|---|---|
| 0 | NULL | No data |
| 1 | External | Data stored in an external block by content ID |
| 3 | Huffman | Canonical Huffman coding |
| 4 | Byte Array Len | Length-prefixed byte array |
| 5 | Byte Array Stop | Stop-byte terminated byte array |
| 6 | Beta | Fixed-width binary coding |
| 7 | Subexponential | Subexponential Golomb coding |
| 8 | Golomb-Rice | Golomb-Rice coding |
| 9 | Gamma | Elias gamma coding |

Each encoding descriptor is stored as: encoding ID (ITF-8) + parameter length (ITF-8) + parameters.

In [ ]:
# Section 4: parse_encoding — reads an encoding descriptor

def parse_encoding(stream: BinaryIO) -> dict:
    """Parse a CRAM encoding descriptor from a stream."""
    codec_id = read_itf8(stream)
    params_len = read_itf8(stream)
    params_data = stream.read(params_len)
    ps = io.BytesIO(params_data)

    if codec_id == 0:  # NULL
        return {'codec': 'NULL'}
    elif codec_id == 1:  # External
        block_content_id = read_itf8(ps)
        return {'codec': 'EXTERNAL', 'content_id': block_content_id}
    elif codec_id == 3:  # Huffman
        num_symbols = read_itf8(ps)
        symbols = [read_itf8(ps) for _ in range(num_symbols)]
        num_lengths = read_itf8(ps)
        bit_lengths = [read_itf8(ps) for _ in range(num_lengths)]
        return {'codec': 'HUFFMAN', 'symbols': symbols, 'bit_lengths': bit_lengths}
    elif codec_id == 4:  # Byte Array Len
        len_encoding = parse_encoding(ps)
        val_encoding = parse_encoding(ps)
        return {'codec': 'BYTE_ARRAY_LEN', 'len_encoding': len_encoding, 'val_encoding': val_encoding}
    elif codec_id == 5:  # Byte Array Stop
        stop_byte = ps.read(1)[0]
        block_content_id = read_itf8(ps)
        return {'codec': 'BYTE_ARRAY_STOP', 'stop_byte': stop_byte, 'content_id': block_content_id}
    elif codec_id == 6:  # Beta
        offset = read_itf8(ps)
        num_bits = read_itf8(ps)
        return {'codec': 'BETA', 'offset': offset, 'num_bits': num_bits}
    elif codec_id == 7:  # Subexponential
        offset = read_itf8(ps)
        k = read_itf8(ps)
        return {'codec': 'SUBEXP', 'offset': offset, 'k': k}
    elif codec_id == 8:  # Golomb-Rice
        offset = read_itf8(ps)
        log2m = read_itf8(ps)
        return {'codec': 'GOLOMB_RICE', 'offset': offset, 'log2m': log2m}
    elif codec_id == 9:  # Gamma
        offset = read_itf8(ps)
        return {'codec': 'GAMMA', 'offset': offset}
    else:
        return {'codec': 'UNKNOWN', 'id': codec_id, 'params': params_data}

print("parse_encoding() defined.")

In [ ]:
# Section 4: CRAMCodec — decodes values using encoding descriptors

def build_huffman_table(symbols: list, bit_lengths: list) -> dict:
    """Build a canonical Huffman decoding table.
    Returns a dict mapping (bit_length, code) -> symbol."""
    if len(symbols) == 1:
        return {(0, 0): symbols[0]}

    pairs = sorted(zip(bit_lengths, symbols))
    table = {}
    code = 0
    prev_length = 0

    for length, symbol in pairs:
        if length == 0:
            table[(0, 0)] = symbol
            continue
        code <<= (length - prev_length)
        table[(length, code)] = symbol
        code += 1
        prev_length = length

    return table


class CRAMCodec:
    """Decodes values from CRAM core and external blocks using encoding descriptors."""

    def __init__(self, encoding: dict, external_blocks: dict):
        self.encoding = encoding
        self.external_blocks = external_blocks
        if encoding['codec'] == 'HUFFMAN':
            self.huffman_table = build_huffman_table(
                encoding['symbols'], encoding['bit_lengths']
            )
            self.max_bits = max(encoding['bit_lengths']) if encoding['bit_lengths'] else 0

    def decode_int(self, core_reader: BitReader) -> int:
        enc = self.encoding
        codec = enc['codec']

        if codec == 'NULL':
            return 0
        elif codec == 'EXTERNAL':
            cid = enc['content_id']
            stream = self.external_blocks.get(cid)
            if stream is None:
                raise CRAMDecodingError(f"External block {cid} not found")
            return read_itf8(stream)
        elif codec == 'HUFFMAN':
            return self._decode_huffman(core_reader)
        elif codec == 'BETA':
            val = core_reader.read_bits(enc['num_bits'])
            return val - enc['offset']
        elif codec == 'GAMMA':
            return self._decode_gamma(core_reader) - enc['offset']
        elif codec == 'SUBEXP':
            return self._decode_subexp(core_reader) - enc['offset']
        elif codec == 'GOLOMB_RICE':
            return self._decode_golomb_rice(core_reader) - enc['offset']
        else:
            raise CRAMDecodingError(f"Cannot decode int with codec {codec}")

    def decode_bytes(self, core_reader: BitReader) -> bytes:
        enc = self.encoding
        codec = enc['codec']

        if codec == 'EXTERNAL':
            cid = enc['content_id']
            stream = self.external_blocks.get(cid)
            if stream is None:
                raise CRAMDecodingError(f"External block {cid} not found")
            # Read a single byte
            b = stream.read(1)
            if not b:
                raise CRAMDecodingError("Unexpected end of external block")
            return b
        elif codec == 'BYTE_ARRAY_LEN':
            len_codec = CRAMCodec(enc['len_encoding'], self.external_blocks)
            val_codec = CRAMCodec(enc['val_encoding'], self.external_blocks)
            length = len_codec.decode_int(core_reader)
            if enc['val_encoding']['codec'] == 'EXTERNAL':
                cid = enc['val_encoding']['content_id']
                stream = self.external_blocks.get(cid)
                if stream is None:
                    raise CRAMDecodingError(f"External block {cid} not found")
                data = stream.read(length)
                if len(data) < length:
                    raise CRAMDecodingError("Short read from external block")
                return data
            else:
                parts = []
                for _ in range(length):
                    parts.append(val_codec.decode_bytes(core_reader))
                return b''.join(parts)
        elif codec == 'BYTE_ARRAY_STOP':
            cid = enc['content_id']
            stop = enc['stop_byte']
            stream = self.external_blocks.get(cid)
            if stream is None:
                raise CRAMDecodingError(f"External block {cid} not found")
            result = bytearray()
            while True:
                b = stream.read(1)
                if not b:
                    break
                if b[0] == stop:
                    break
                result.append(b[0])
            return bytes(result)
        elif codec == 'HUFFMAN':
            return bytes([self._decode_huffman(core_reader)])
        else:
            raise CRAMDecodingError(f"Cannot decode bytes with codec {codec}")

    def decode_byte(self, core_reader: BitReader) -> int:
        enc = self.encoding
        codec = enc['codec']
        if codec == 'EXTERNAL':
            cid = enc['content_id']
            stream = self.external_blocks.get(cid)
            if stream is None:
                raise CRAMDecodingError(f"External block {cid} not found")
            b = stream.read(1)
            if not b:
                raise CRAMDecodingError("Unexpected end of external block")
            return b[0]
        elif codec == 'HUFFMAN':
            return self._decode_huffman(core_reader)
        elif codec == 'BETA':
            val = core_reader.read_bits(enc['num_bits'])
            return val - enc['offset']
        else:
            return self.decode_int(core_reader)

    def _decode_huffman(self, reader: BitReader) -> int:
        if len(self.encoding['symbols']) == 1:
            return self.encoding['symbols'][0]
        code = 0
        length = 0
        for _ in range(self.max_bits + 1):
            code = (code << 1) | reader.read_bit()
            length += 1
            key = (length, code)
            if key in self.huffman_table:
                return self.huffman_table[key]
        raise CRAMDecodingError("Huffman decode failed: no matching code")

    def _decode_gamma(self, reader: BitReader) -> int:
        n = 0
        while reader.read_bit() == 0:
            n += 1
        val = 1
        for _ in range(n):
            val = (val << 1) | reader.read_bit()
        return val

    def _decode_subexp(self, reader: BitReader) -> int:
        k = self.encoding['k']
        n = 0
        while reader.read_bit() == 1:
            n += 1
        if n == 0:
            return reader.read_bits(k)
        else:
            return (1 << (n + k - 1)) + reader.read_bits(n + k - 1)

    def _decode_golomb_rice(self, reader: BitReader) -> int:
        log2m = self.encoding['log2m']
        q = 0
        while reader.read_bit() == 0:
            q += 1
        r = reader.read_bits(log2m)
        return (q << log2m) | r

print("CRAMCodec and build_huffman_table() defined.")

In [ ]:
# Section 4: Encoding scheme tests

def test_huffman():
    # Single symbol Huffman: always returns that symbol
    table = build_huffman_table([42], [0])
    assert table == {(0, 0): 42}

    # Two symbols: A=0 (1 bit), B=1 (1 bit)
    table2 = build_huffman_table([65, 66], [1, 1])
    assert (1, 0) in table2 and table2[(1, 0)] == 65
    assert (1, 1) in table2 and table2[(1, 1)] == 66

    # Test Huffman via CRAMCodec
    enc = {'codec': 'HUFFMAN', 'symbols': [65, 66], 'bit_lengths': [1, 1]}
    codec = CRAMCodec(enc, {})
    reader = BitReader(b'\x80')  # 10000000 -> first bit=1 -> B, second bit=0 -> A
    assert codec.decode_int(reader) == 66  # bit=1 -> B
    assert codec.decode_int(reader) == 65  # bit=0 -> A
    print("Huffman tests passed.")

def test_beta():
    enc = {'codec': 'BETA', 'offset': 0, 'num_bits': 8}
    codec = CRAMCodec(enc, {})
    reader = BitReader(b'\xFF')
    assert codec.decode_int(reader) == 255
    print("Beta test passed.")

def test_gamma():
    enc = {'codec': 'GAMMA', 'offset': 0}
    codec = CRAMCodec(enc, {})
    # Gamma(1) = 1 (single '1' bit)
    reader = BitReader(b'\x80')  # 10000000
    assert codec.decode_int(reader) == 1
    # Gamma(2) = 010 -> n=1, then read 1 bit: 01[0] -> 2
    reader2 = BitReader(b'\x40')  # 01000000
    assert codec.decode_int(reader2) == 2
    print("Gamma test passed.")

test_huffman()
test_beta()
test_gamma()

## Section 5: File Header

The CRAM file definition is a 26-byte fixed header:
- Bytes 0-3: Magic number `CRAM` (0x43 0x52 0x41 0x4D)
- Byte 4: Major version (3 for CRAM v3.0)
- Byte 5: Minor version (0 or 1)
- Bytes 6-25: File ID (20 bytes, typically a UUID or hash)

In [ ]:
# Section 5: File header parser

CRAM_MAGIC = b'CRAM'

def parse_file_header(stream: BinaryIO) -> dict:
    """Parse the 26-byte CRAM file definition header."""
    magic = stream.read(4)
    if magic != CRAM_MAGIC:
        raise CRAMError(f"Not a CRAM file: magic bytes = {magic!r}")

    major = struct.unpack('B', stream.read(1))[0]
    minor = struct.unpack('B', stream.read(1))[0]

    if major != 3:
        raise CRAMVersionError(f"Unsupported CRAM version: {major}.{minor} (only v3.x supported)")

    file_id = stream.read(20)

    return {
        'magic': magic.decode('ascii'),
        'major_version': major,
        'minor_version': minor,
        'file_id': file_id.hex()
    }

print("parse_file_header() defined.")

## Section 6: Container & Block Parsing

A CRAM file is a sequence of **containers**. Each container has:
- A header with metadata (length, ref ID, alignment start/span, record count, landmarks, CRC32)
- One or more **blocks** (compression header block, core data block, external blocks)

Each block has:
- Compression method, content type, content ID
- Compressed and uncompressed sizes
- Raw data (decompressed on read)

Content types: 0=FILE_HEADER, 1=COMPRESSION_HEADER, 2=MAPPED_SLICE_HEADER, 3=RESERVED, 4=EXTERNAL_DATA, 5=CORE_DATA

In [ ]:
# Section 6: Container header parser

def parse_container_header(stream: BinaryIO) -> Optional[dict]:
    """Parse a CRAM container header. Returns None at EOF."""
    # Container length (int32)
    length_bytes = stream.read(4)
    if len(length_bytes) < 4:
        return None
    container_length = struct.unpack('<i', length_bytes)[0]

    ref_seq_id = read_itf8(stream)
    start_pos = read_itf8(stream)
    align_span = read_itf8(stream)
    num_records = read_itf8(stream)
    record_counter = read_ltf8(stream)
    num_bases = read_ltf8(stream)
    num_blocks = read_itf8(stream)
    num_landmarks = read_itf8(stream)
    landmarks = [read_itf8(stream) for _ in range(num_landmarks)]

    crc32 = struct.unpack('<I', stream.read(4))[0]

    return {
        'length': container_length,
        'ref_seq_id': ref_seq_id,
        'start_pos': start_pos,
        'align_span': align_span,
        'num_records': num_records,
        'record_counter': record_counter,
        'num_bases': num_bases,
        'num_blocks': num_blocks,
        'num_landmarks': num_landmarks,
        'landmarks': landmarks,
        'crc32': crc32
    }


def is_eof_container(header: dict) -> bool:
    """Check if this container header represents the EOF marker."""
    return (header['length'] == 15 and
            header['ref_seq_id'] == -1 and
            header['start_pos'] == 4542278 and
            header['align_span'] == 0 and
            header['num_records'] == 0 and
            header['num_blocks'] == 0 and
            header['num_landmarks'] == 0)

print("Container header parser defined.")

In [ ]:
# Section 6: Block parser

CONTENT_TYPES = {
    0: 'FILE_HEADER',
    1: 'COMPRESSION_HEADER',
    2: 'MAPPED_SLICE_HEADER',
    3: 'RESERVED',
    4: 'EXTERNAL_DATA',
    5: 'CORE_DATA'
}

COMPRESSION_METHODS = {
    0: 'RAW',
    1: 'GZIP',
    2: 'BZIP2',
    3: 'LZMA',
    4: 'RANS'
}

def parse_block(stream: BinaryIO) -> dict:
    """Parse a single CRAM block, decompressing its data."""
    method = stream.read(1)
    if not method:
        raise CRAMError("Unexpected end of stream reading block")
    method = method[0]

    content_type = read_itf8(stream)
    content_id = read_itf8(stream)
    compressed_size = read_itf8(stream)
    uncompressed_size = read_itf8(stream)

    compressed_data = stream.read(compressed_size)
    if len(compressed_data) < compressed_size:
        raise CRAMError(f"Short read: got {len(compressed_data)}, expected {compressed_size}")

    crc32_val = struct.unpack('<I', stream.read(4))[0]

    # Decompress
    data = decompress_block(method, compressed_data, uncompressed_size)

    return {
        'method': method,
        'method_name': COMPRESSION_METHODS.get(method, f'UNKNOWN({method})'),
        'content_type': content_type,
        'content_type_name': CONTENT_TYPES.get(content_type, f'UNKNOWN({content_type})'),
        'content_id': content_id,
        'compressed_size': compressed_size,
        'uncompressed_size': uncompressed_size,
        'data': data,
        'crc32': crc32_val
    }

print("parse_block() defined.")

## Section 7: SAM Header

The first container in a CRAM file holds the SAM header as plain text. This contains
`@HD` (header), `@SQ` (sequence dictionary), `@RG` (read group), and `@PG` (program)
lines that describe the alignment metadata.

In [ ]:
# Section 7: SAM header parser

def parse_sam_header(text: str) -> dict:
    """Parse SAM header text into structured dict."""
    result = {
        'HD': {},
        'SQ': [],
        'RG': [],
        'PG': [],
        'CO': [],
        'raw': text
    }

    for line in text.strip().split('\n'):
        if not line.startswith('@'):
            continue
        parts = line.split('\t')
        tag = parts[0][1:]  # Remove @

        if tag == 'HD':
            for field in parts[1:]:
                if ':' in field:
                    k, v = field.split(':', 1)
                    result['HD'][k] = v
        elif tag == 'SQ':
            entry = {}
            for field in parts[1:]:
                if ':' in field:
                    k, v = field.split(':', 1)
                    entry[k] = v
            result['SQ'].append(entry)
        elif tag == 'RG':
            entry = {}
            for field in parts[1:]:
                if ':' in field:
                    k, v = field.split(':', 1)
                    entry[k] = v
            result['RG'].append(entry)
        elif tag == 'PG':
            entry = {}
            for field in parts[1:]:
                if ':' in field:
                    k, v = field.split(':', 1)
                    entry[k] = v
            result['PG'].append(entry)
        elif tag == 'CO':
            result['CO'].append('\t'.join(parts[1:]))

    return result


def read_sam_header_container(stream: BinaryIO) -> dict:
    """Read the SAM header from the first CRAM container."""
    container_header = parse_container_header(stream)
    if container_header is None:
        raise CRAMError("No SAM header container found")

    block = parse_block(stream)
    header_data = block['data']

    # The SAM header block contains: int32 header_length + header_text
    header_stream = io.BytesIO(header_data)
    header_length = read_itf8(header_stream)
    header_text = header_stream.read(header_length).decode('utf-8', errors='replace')

    return parse_sam_header(header_text)

print("SAM header parser defined.")

## Section 8: Compression Header

The compression header is the first block in each data container. It contains three maps:

1. **Preservation map** — flags like `RN` (read names included), `AP` (AP-delta), `RR` (reference required),
   `SM` (substitution matrix), `TD` (tag dictionary)
2. **Data series encoding map** — encoding descriptors for each data series (BF, CF, RI, RL, AP, RG, RN, etc.)
3. **Tag encoding map** — encoding descriptors for auxiliary tags, keyed by 3-byte packed tag+type identifiers

In [ ]:
# Section 8: Compression header parser

def parse_compression_header(block_data: bytes) -> dict:
    """Parse a CRAM compression header block."""
    stream = io.BytesIO(block_data)

    # --- Preservation map ---
    pmap_size = read_itf8(stream)
    pmap_count = read_itf8(stream)

    preservation = {
        'read_names_included': True,
        'AP_delta': True,
        'reference_required': True,
        'substitution_matrix': None,
        'tag_dictionary': []
    }

    for _ in range(pmap_count):
        key = stream.read(2).decode('ascii')

        if key == 'RN':
            preservation['read_names_included'] = (stream.read(1)[0] != 0)
        elif key == 'AP':
            preservation['AP_delta'] = (stream.read(1)[0] != 0)
        elif key == 'RR':
            preservation['reference_required'] = (stream.read(1)[0] != 0)
        elif key == 'SM':
            # Substitution matrix: 5 bytes, each with 4 2-bit codes
            sm_data = stream.read(5)
            bases = 'ACGTN'
            matrix = {}
            for i, ref_base in enumerate(bases):
                byte_val = sm_data[i]
                subs = []
                for j in range(4):
                    code = (byte_val >> (6 - 2 * j)) & 0x03
                    subs.append(code)
                matrix[ref_base] = subs
            preservation['substitution_matrix'] = matrix
        elif key == 'TD':
            # Tag dictionary: ITF-8 length + data
            td_len = read_itf8(stream)
            td_data = stream.read(td_len)
            # Parse tag dictionary entries (NUL-separated, double-NUL ends a row)
            tag_dict = []
            current_row = []
            i = 0
            while i < len(td_data):
                if td_data[i] == 0:
                    tag_dict.append(current_row)
                    current_row = []
                    i += 1
                else:
                    if i + 2 < len(td_data):
                        tag = chr(td_data[i]) + chr(td_data[i+1])
                        tag_type = chr(td_data[i+2])
                        current_row.append((tag, tag_type))
                        i += 3
                    else:
                        break
            if current_row:
                tag_dict.append(current_row)
            preservation['tag_dictionary'] = tag_dict
        else:
            # Unknown key, read one byte
            stream.read(1)

    # --- Data series encoding map ---
    ds_map_size = read_itf8(stream)
    ds_map_count = read_itf8(stream)
    data_series = {}

    for _ in range(ds_map_count):
        key = stream.read(2).decode('ascii')
        encoding = parse_encoding(stream)
        data_series[key] = encoding

    # --- Tag encoding map ---
    tag_map_size = read_itf8(stream)
    tag_map_count = read_itf8(stream)
    tag_encodings = {}

    for _ in range(tag_map_count):
        # Key is packed as ITF-8: (tag[0] << 16) | (tag[1] << 8) | type_char
        key_int = read_itf8(stream)
        tag_char1 = chr((key_int >> 16) & 0xFF)
        tag_char2 = chr((key_int >> 8) & 0xFF)
        type_char = chr(key_int & 0xFF)
        tag_key = f"{tag_char1}{tag_char2}{type_char}"
        encoding = parse_encoding(stream)
        tag_encodings[tag_key] = encoding

    return {
        'preservation': preservation,
        'data_series': data_series,
        'tag_encodings': tag_encodings
    }

print("parse_compression_header() defined.")

## Section 9: Slice & Record Decoding

Each data container has one or more **slices**. Each slice contains:
- A **slice header block** (ref ID, alignment start/span, record count, content block IDs, ref MD5)
- A **core data block** (bit-packed fields read via BitReader)
- **External data blocks** (keyed by content ID, read as byte streams)

The record decoder reads each field using the encoding specified in the compression header:
- Bit flags (BF), compression flags (CF), ref ID (RI), read length (RL), alignment position (AP)
- Read name (RN), mate info, mapping quality (MQ)
- Read features (substitutions, insertions, deletions, clips, etc.)
- Tags (via tag dictionary + tag encoding map), quality scores (QS)

In [ ]:
# Section 9: Slice header parser

def parse_slice_header(block_data: bytes) -> dict:
    """Parse a CRAM slice header block."""
    stream = io.BytesIO(block_data)

    ref_seq_id = read_itf8(stream)
    align_start = read_itf8(stream)
    align_span = read_itf8(stream)
    num_records = read_itf8(stream)
    record_counter = read_ltf8(stream)
    num_blocks = read_itf8(stream)
    num_content_ids = read_itf8(stream)
    content_ids = [read_itf8(stream) for _ in range(num_content_ids)]
    embedded_ref_content_id = read_itf8(stream)
    ref_md5 = stream.read(16)

    # Optional tags (if data remains)
    optional_tags = None
    remaining = stream.read()
    if remaining:
        optional_tags = remaining

    return {
        'ref_seq_id': ref_seq_id,
        'align_start': align_start,
        'align_span': align_span,
        'num_records': num_records,
        'record_counter': record_counter,
        'num_blocks': num_blocks,
        'content_ids': content_ids,
        'embedded_ref_content_id': embedded_ref_content_id,
        'ref_md5': ref_md5.hex(),
        'optional_tags': optional_tags
    }

print("parse_slice_header() defined.")

In [ ]:
# Section 9: Tag value decoder

BAM_TAG_SIZES = {
    'A': 1, 'c': 1, 'C': 1, 's': 2, 'S': 2, 'i': 4, 'I': 4, 'f': 4
}

def _decode_tag_value(tag_type: str, data_stream: BinaryIO) -> Any:
    """Decode a BAM-style tag value from a byte stream."""
    if tag_type == 'A':
        return chr(data_stream.read(1)[0])
    elif tag_type == 'c':
        return struct.unpack('<b', data_stream.read(1))[0]
    elif tag_type == 'C':
        return struct.unpack('<B', data_stream.read(1))[0]
    elif tag_type == 's':
        return struct.unpack('<h', data_stream.read(2))[0]
    elif tag_type == 'S':
        return struct.unpack('<H', data_stream.read(2))[0]
    elif tag_type == 'i':
        return struct.unpack('<i', data_stream.read(4))[0]
    elif tag_type == 'I':
        return struct.unpack('<I', data_stream.read(4))[0]
    elif tag_type == 'f':
        return struct.unpack('<f', data_stream.read(4))[0]
    elif tag_type == 'Z':
        result = bytearray()
        while True:
            b = data_stream.read(1)
            if not b or b[0] == 0:
                break
            result.append(b[0])
        return result.decode('utf-8', errors='replace')
    elif tag_type == 'H':
        result = bytearray()
        while True:
            b = data_stream.read(1)
            if not b or b[0] == 0:
                break
            result.append(b[0])
        return result.hex()
    elif tag_type == 'B':
        sub_type = chr(data_stream.read(1)[0])
        count = struct.unpack('<I', data_stream.read(4))[0]
        fmt_map = {'c': '<b', 'C': '<B', 's': '<h', 'S': '<H',
                   'i': '<i', 'I': '<I', 'f': '<f'}
        fmt = fmt_map.get(sub_type, '<B')
        size = struct.calcsize(fmt)
        values = []
        for _ in range(count):
            values.append(struct.unpack(fmt, data_stream.read(size))[0])
        return values
    else:
        return data_stream.read(1)

print("_decode_tag_value() defined.")

In [ ]:
# Section 9: Sequence reconstruction

def _get_substitution_base(ref_base: str, code: int, sub_matrix: dict) -> str:
    """Get the substituted base given a reference base and substitution code."""
    bases = 'ACGTN'
    if sub_matrix and ref_base in sub_matrix:
        codes = sub_matrix[ref_base]
        # The substitution code indexes into the alternatives (bases other than ref)
        alt_bases = [b for b in bases if b != ref_base]
        if code < len(alt_bases):
            return alt_bases[code]
    # Fallback
    alt_bases = [b for b in bases if b != ref_base]
    if code < len(alt_bases):
        return alt_bases[code]
    return 'N'


def _reconstruct_sequence(read_length: int, features: list,
                          ref_seq: Optional[str], align_start: int) -> str:
    """Reconstruct read sequence from reference + features."""
    if ref_seq is None:
        # Unmapped or no reference: build from features only
        seq = ['N'] * read_length
        for feat in features:
            fc = feat['code']
            pos = feat['pos'] - 1  # 1-based to 0-based
            if fc == 'B':  # Base (read base)
                if 0 <= pos < read_length:
                    seq[pos] = chr(feat['base'])
            elif fc == 'I':  # Insertion
                pass  # handled in CIGAR, bases already in features
            elif fc == 'S':  # Soft clip
                bases = feat.get('bases', b'')
                for j, b in enumerate(bases):
                    if 0 <= pos + j < read_length:
                        seq[pos + j] = chr(b) if isinstance(b, int) else b
            elif fc == 'b':  # Bases (stretch of bases)
                bases = feat.get('bases', b'')
                for j, b in enumerate(bases):
                    if 0 <= pos + j < read_length:
                        seq[pos + j] = chr(b) if isinstance(b, int) else b
            elif fc == 'i':  # Single-base insertion
                if 0 <= pos < read_length:
                    seq[pos] = chr(feat['base'])
        return ''.join(seq)

    # Mapped: start with reference bases, then apply features
    ref_offset = align_start - 1  # Convert to 0-based
    seq = list('N' * read_length)

    # Copy reference bases first
    read_pos = 0
    ref_pos = ref_offset
    for i in range(read_length):
        if 0 <= ref_pos < len(ref_seq):
            seq[i] = ref_seq[ref_pos].upper()
        ref_pos += 1

    # Apply features
    for feat in features:
        fc = feat['code']
        pos = feat['pos'] - 1  # 1-based to 0-based within read

        if fc == 'X':  # Substitution
            if 0 <= pos < read_length:
                seq[pos] = feat['sub_base']
        elif fc == 'B':  # Read base
            if 0 <= pos < read_length:
                seq[pos] = chr(feat['base'])
        elif fc == 'S':  # Soft clip
            bases = feat.get('bases', b'')
            for j, b in enumerate(bases):
                if 0 <= pos + j < read_length:
                    seq[pos + j] = chr(b) if isinstance(b, int) else b
        elif fc == 'I':  # Insertion
            bases = feat.get('bases', b'')
            for j, b in enumerate(bases):
                if 0 <= pos + j < read_length:
                    seq[pos + j] = chr(b) if isinstance(b, int) else b
        elif fc == 'i':  # Single-base insertion
            if 0 <= pos < read_length:
                seq[pos] = chr(feat['base'])
        elif fc == 'b':  # Bases
            bases = feat.get('bases', b'')
            for j, b in enumerate(bases):
                if 0 <= pos + j < read_length:
                    seq[pos + j] = chr(b) if isinstance(b, int) else b

    return ''.join(seq)

print("_reconstruct_sequence() defined.")

In [ ]:
# Section 9: Record decoder

# SAM flag constants
BAM_FPAIRED = 0x1
BAM_FPROPER_PAIR = 0x2
BAM_FUNMAP = 0x4
BAM_FMUNMAP = 0x8
BAM_FREVERSE = 0x10
BAM_FMREVERSE = 0x20
BAM_FREAD1 = 0x40
BAM_FREAD2 = 0x80
BAM_FSECONDARY = 0x100
BAM_FQCFAIL = 0x200
BAM_FDUP = 0x400
BAM_FSUPPLEMENTARY = 0x800

# CRAM compression flag bits
CF_PRESERVE_QUAL = 0x01
CF_DETACHED = 0x02
CF_MATE_DOWNSTREAM = 0x04

def decode_records(slice_header: dict, comp_header: dict,
                   core_data: bytes, external_blocks: dict,
                   ref_seqs: Optional[dict] = None,
                   sam_header: Optional[dict] = None) -> list:
    """Decode alignment records from a CRAM slice."""
    preservation = comp_header['preservation']
    ds = comp_header['data_series']
    tag_encs = comp_header['tag_encodings']
    tag_dict = preservation.get('tag_dictionary', [])
    sub_matrix = preservation.get('substitution_matrix')
    ap_delta = preservation.get('AP_delta', True)
    rn_included = preservation.get('read_names_included', True)

    # Wrap external block data as BytesIO streams
    ext_streams = {}
    for cid, data in external_blocks.items():
        if isinstance(data, bytes):
            ext_streams[cid] = io.BytesIO(data)
        else:
            ext_streams[cid] = data

    # Create codecs for each data series
    codecs = {}
    for key, enc in ds.items():
        codecs[key] = CRAMCodec(enc, ext_streams)

    # Create codecs for tag encodings
    tag_codecs = {}
    for key, enc in tag_encs.items():
        tag_codecs[key] = CRAMCodec(enc, ext_streams)

    # Core data BitReader
    core_reader = BitReader(core_data)

    # Get reference sequence for this slice
    ref_seq = None
    if ref_seqs and slice_header['ref_seq_id'] >= 0:
        if sam_header and sam_header.get('SQ'):
            sq_list = sam_header['SQ']
            ref_id = slice_header['ref_seq_id']
            if ref_id < len(sq_list):
                ref_name = sq_list[ref_id].get('SN', '')
                ref_seq = ref_seqs.get(ref_name)

    records = []
    prev_pos = slice_header['align_start']

    for rec_idx in range(slice_header['num_records']):
        try:
            record = _decode_single_record(
                codecs, tag_codecs, tag_dict, core_reader, ext_streams,
                preservation, ap_delta, rn_included, prev_pos,
                ref_seq, sub_matrix, slice_header, sam_header
            )
            prev_pos = record.get('align_start', prev_pos)
            records.append(record)
        except Exception as e:
            print(f"  Warning: Error decoding record {rec_idx}: {e}")
            records.append({'error': str(e), 'record_index': rec_idx})
            break

    return records


def _decode_single_record(codecs, tag_codecs, tag_dict, core_reader,
                          ext_streams, preservation, ap_delta, rn_included,
                          prev_pos, ref_seq, sub_matrix, slice_header, sam_header):
    """Decode a single CRAM record."""
    record = {}

    # Bit flags (BAM flags)
    bf = codecs['BF'].decode_int(core_reader)
    record['flags'] = bf

    # Compression flags
    cf = codecs['CF'].decode_int(core_reader)
    record['cram_flags'] = cf

    # Reference ID (if multi-ref slice)
    if slice_header['ref_seq_id'] == -2:
        ri = codecs['RI'].decode_int(core_reader)
        record['ref_id'] = ri
    else:
        record['ref_id'] = slice_header['ref_seq_id']

    # Read length
    rl = codecs['RL'].decode_int(core_reader)
    record['read_length'] = rl

    # Alignment position
    if ap_delta:
        ap = codecs['AP'].decode_int(core_reader)
        record['align_start'] = prev_pos + ap
    else:
        ap = codecs['AP'].decode_int(core_reader)
        record['align_start'] = ap

    # Read group
    rg = codecs['RG'].decode_int(core_reader)
    record['read_group'] = rg

    # Read name
    if rn_included:
        rn = codecs['RN'].decode_bytes(core_reader)
        record['read_name'] = rn.decode('utf-8', errors='replace')
    else:
        record['read_name'] = str(slice_header.get('record_counter', 0))

    # Mate information
    if bf & BAM_FPAIRED:
        if cf & CF_DETACHED:
            # Detached mate: full mate info
            mf = codecs['MF'].decode_int(core_reader)
            record['mate_flags'] = mf

            if not rn_included:
                mn = codecs['RN'].decode_bytes(core_reader)
                record['mate_read_name'] = mn.decode('utf-8', errors='replace')

            ns = codecs['NS'].decode_int(core_reader)
            record['mate_ref_id'] = ns

            np_val = codecs['NP'].decode_int(core_reader)
            record['mate_align_start'] = np_val

            ts = codecs['TS'].decode_int(core_reader)
            record['template_size'] = ts
        elif cf & CF_MATE_DOWNSTREAM:
            nf = codecs['NF'].decode_int(core_reader) if 'NF' in codecs else 0
            record['records_to_mate'] = nf

    # Mapping quality
    if not (bf & BAM_FUNMAP):
        mq = codecs['MQ'].decode_int(core_reader)
        record['mapping_quality'] = mq
    else:
        record['mapping_quality'] = 0

    # Read features (for mapped reads)
    features = []
    if not (bf & BAM_FUNMAP):
        fn = codecs['FN'].decode_int(core_reader)
        feat_pos = 0

        for _ in range(fn):
            fc = codecs['FC'].decode_byte(core_reader)
            fc_char = chr(fc) if isinstance(fc, int) else fc

            fp = codecs['FP'].decode_int(core_reader)
            feat_pos += fp

            feat = {'code': fc_char, 'pos': feat_pos}

            if fc_char == 'X':  # Substitution
                bs = codecs['BS'].decode_int(core_reader)
                # Get reference base at this position
                ref_base = 'N'
                if ref_seq:
                    ref_idx = record['align_start'] - 1 + feat_pos - 1
                    if 0 <= ref_idx < len(ref_seq):
                        ref_base = ref_seq[ref_idx].upper()
                feat['sub_code'] = bs
                feat['sub_base'] = _get_substitution_base(ref_base, bs, sub_matrix)
            elif fc_char == 'I':  # Insertion (multi-base)
                ins_data = codecs['IN'].decode_bytes(core_reader)
                feat['bases'] = ins_data
            elif fc_char == 'S':  # Soft clip
                sc_data = codecs['SC'].decode_bytes(core_reader)
                feat['bases'] = sc_data
            elif fc_char == 'D':  # Deletion
                dl = codecs['DL'].decode_int(core_reader)
                feat['length'] = dl
            elif fc_char == 'i':  # Single-base insertion
                ba = codecs['BA'].decode_byte(core_reader)
                feat['base'] = ba
            elif fc_char == 'B':  # Read base + quality
                ba = codecs['BA'].decode_byte(core_reader)
                qs = codecs['QS'].decode_byte(core_reader)
                feat['base'] = ba
                feat['quality'] = qs
            elif fc_char == 'Q':  # Quality score only
                qs = codecs['QS'].decode_byte(core_reader)
                feat['quality'] = qs
            elif fc_char == 'N':  # Reference skip
                rs = codecs['RS'].decode_int(core_reader) if 'RS' in codecs else 0
                feat['length'] = rs
            elif fc_char == 'H':  # Hard clip
                hc = codecs['HC'].decode_int(core_reader) if 'HC' in codecs else 0
                feat['length'] = hc
            elif fc_char == 'P':  # Padding
                pd = codecs['PD'].decode_int(core_reader) if 'PD' in codecs else 0
                feat['length'] = pd
            elif fc_char == 'b':  # Bases
                bb_data = codecs['BB'].decode_bytes(core_reader) if 'BB' in codecs else b''
                feat['bases'] = bb_data
            elif fc_char == 'q':  # Quality scores
                qq_data = codecs['QQ'].decode_bytes(core_reader) if 'QQ' in codecs else b''
                feat['qualities'] = qq_data

            features.append(feat)

    record['features'] = features

    # Reconstruct sequence
    record['sequence'] = _reconstruct_sequence(
        rl, features, ref_seq, record['align_start']
    )

    # Tags
    tl = codecs['TL'].decode_int(core_reader)
    tags = {}
    if tl < len(tag_dict):
        tag_list = tag_dict[tl]
        for tag_name, tag_type in tag_list:
            tag_key = f"{tag_name}{tag_type}"
            if tag_key in tag_codecs:
                tag_data = tag_codecs[tag_key].decode_bytes(core_reader)
                tag_stream = io.BytesIO(tag_data)
                tag_value = _decode_tag_value(tag_type, tag_stream)
                tags[tag_name] = {'type': tag_type, 'value': tag_value}
            else:
                tags[tag_name] = {'type': tag_type, 'value': None}
    record['tags'] = tags

    # Quality scores
    if cf & CF_PRESERVE_QUAL:
        quals = []
        for _ in range(rl):
            q = codecs['QS'].decode_byte(core_reader)
            quals.append(q)
        record['quality'] = quals
    else:
        record['quality'] = [255] * rl

    return record

print("decode_records() and _decode_single_record() defined.")

## Section 10: Top-Level Parser

The main parser orchestrates:
1. Reading the file header (26 bytes)
2. Reading the SAM header container
3. Iterating through data containers, each containing:
   - A compression header block
   - One or more slices (each with header + core + external blocks)
4. Decoding records from each slice using the compression header

An optional FASTA reference loader provides reference sequences for mapped-read reconstruction.

In [ ]:
# Section 10: FASTA reference loader

def _load_reference(fasta_path: str) -> dict:
    """Load a FASTA reference file into a {name: sequence} dict."""
    refs = {}
    current_name = None
    current_seq = []

    with open(fasta_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_name is not None:
                    refs[current_name] = ''.join(current_seq)
                current_name = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)

    if current_name is not None:
        refs[current_name] = ''.join(current_seq)

    return refs

print("_load_reference() defined.")

In [ ]:
# Section 10: Main CRAM parser

def parse_cram(filepath: str, reference_path: Optional[str] = None,
               max_containers: int = 100, max_records: int = 10000) -> dict:
    """Parse a CRAM v3.0 file and return decoded records.

    Args:
        filepath: Path to the .cram file
        reference_path: Optional path to FASTA reference
        max_containers: Maximum number of data containers to parse
        max_records: Maximum total records to decode

    Returns:
        Dict with 'file_header', 'sam_header', 'records', 'container_summaries'
    """
    ref_seqs = None
    if reference_path and os.path.exists(reference_path):
        print(f"Loading reference from {reference_path}...")
        ref_seqs = _load_reference(reference_path)
        print(f"  Loaded {len(ref_seqs)} reference sequences.")

    result = {
        'file_header': None,
        'sam_header': None,
        'records': [],
        'container_summaries': []
    }

    with open(filepath, 'rb') as f:
        # 1. File header
        print("Reading file header...")
        file_header = parse_file_header(f)
        result['file_header'] = file_header
        print(f"  CRAM v{file_header['major_version']}.{file_header['minor_version']}")
        print(f"  File ID: {file_header['file_id']}")

        # 2. SAM header
        print("Reading SAM header...")
        sam_header = read_sam_header_container(f)
        result['sam_header'] = sam_header
        print(f"  {len(sam_header.get('SQ', []))} reference sequences")
        print(f"  {len(sam_header.get('RG', []))} read groups")

        # 3. Data containers
        container_count = 0
        total_records = 0

        while container_count < max_containers and total_records < max_records:
            container_start = f.tell()
            container_header = parse_container_header(f)
            if container_header is None:
                print("End of file reached.")
                break
            if is_eof_container(container_header):
                print("EOF container found.")
                break

            container_count += 1
            summary = {
                'index': container_count,
                'ref_seq_id': container_header['ref_seq_id'],
                'start_pos': container_header['start_pos'],
                'align_span': container_header['align_span'],
                'num_records': container_header['num_records'],
                'num_blocks': container_header['num_blocks'],
                'offset': container_start
            }
            result['container_summaries'].append(summary)
            print(f"\nContainer {container_count}: ref={container_header['ref_seq_id']}, "
                  f"pos={container_header['start_pos']}, "
                  f"records={container_header['num_records']}, "
                  f"blocks={container_header['num_blocks']}")

            try:
                # Read compression header block
                comp_block = parse_block(f)
                if comp_block['content_type'] != 1:
                    print(f"  Warning: Expected compression header (type 1), got type {comp_block['content_type']}")
                    # Skip remaining blocks
                    for _ in range(container_header['num_blocks'] - 1):
                        parse_block(f)
                    continue

                comp_header = parse_compression_header(comp_block['data'])
                print(f"  Compression header: {len(comp_header['data_series'])} data series, "
                      f"{len(comp_header['tag_encodings'])} tag encodings")

                # Read remaining blocks (slices)
                remaining_blocks = container_header['num_blocks'] - 1
                blocks = []
                for _ in range(remaining_blocks):
                    block = parse_block(f)
                    blocks.append(block)

                # Group blocks into slices
                slice_idx = 0
                block_idx = 0
                while block_idx < len(blocks):
                    block = blocks[block_idx]

                    if block['content_type'] == 2:  # Slice header
                        slice_header = parse_slice_header(block['data'])
                        print(f"  Slice {slice_idx}: ref={slice_header['ref_seq_id']}, "
                              f"start={slice_header['align_start']}, "
                              f"records={slice_header['num_records']}, "
                              f"blocks={slice_header['num_blocks']}")

                        # Collect core and external blocks for this slice
                        core_data = None
                        external_blocks = {}
                        for j in range(1, slice_header['num_blocks'] + 1):
                            if block_idx + j < len(blocks):
                                b = blocks[block_idx + j]
                                if b['content_type'] == 5:  # Core
                                    core_data = b['data']
                                elif b['content_type'] == 4:  # External
                                    external_blocks[b['content_id']] = b['data']

                        if core_data is not None:
                            # Decode records
                            slice_records = decode_records(
                                slice_header, comp_header,
                                core_data, external_blocks,
                                ref_seqs, sam_header
                            )
                            valid = [r for r in slice_records if 'error' not in r]
                            print(f"    Decoded {len(valid)}/{slice_header['num_records']} records")
                            result['records'].extend(slice_records)
                            total_records += len(slice_records)
                        else:
                            print(f"    Warning: No core data block found for slice {slice_idx}")

                        block_idx += slice_header['num_blocks'] + 1
                        slice_idx += 1
                    else:
                        block_idx += 1

            except Exception as e:
                print(f"  Error processing container {container_count}: {e}")
                import traceback
                traceback.print_exc()

    print(f"\nParsing complete: {len(result['records'])} total records from {container_count} containers.")
    return result

print("parse_cram() defined.")

## Section 11: CRAI Index

The `.crai` file is a gzip-compressed, tab-delimited index for random access into CRAM files.
Each line contains 6 fields:
1. **seq_id** — reference sequence ID
2. **start** — alignment start position
3. **span** — alignment span
4. **container_offset** — byte offset of the container in the CRAM file
5. **slice_offset** — byte offset of the slice within the container
6. **slice_size** — size of the slice in bytes

In [ ]:
# Section 11: CRAI parser

def parse_crai(filepath: str) -> list:
    """Parse a .crai index file (gzip-compressed tab-delimited).

    Returns a list of dicts with keys:
    seq_id, start, span, container_offset, slice_offset, slice_size
    """
    import gzip as gzip_mod

    entries = []
    with gzip_mod.open(filepath, 'rt') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            if len(parts) >= 6:
                entries.append({
                    'seq_id': int(parts[0]),
                    'start': int(parts[1]),
                    'span': int(parts[2]),
                    'container_offset': int(parts[3]),
                    'slice_offset': int(parts[4]),
                    'slice_size': int(parts[5])
                })

    return entries


def random_access_query(crai_entries: list, seq_id: int,
                        start: int, end: int) -> list:
    """Find CRAI index entries overlapping a genomic region.

    Args:
        crai_entries: Parsed CRAI index entries
        seq_id: Reference sequence ID to query
        start: Start position (1-based)
        end: End position (1-based)

    Returns:
        List of matching CRAI entries
    """
    matches = []
    for entry in crai_entries:
        if entry['seq_id'] != seq_id:
            continue
        entry_start = entry['start']
        entry_end = entry['start'] + entry['span']
        # Check overlap
        if entry_start < end and entry_end > start:
            matches.append(entry)
    return matches

print("parse_crai() and random_access_query() defined.")

## Section 12: Output Formatting & Demo

This section provides:
- **SAM output formatting** — converting decoded records to SAM text lines
- **CIGAR string building** — reconstructing CIGAR from read features
- **Sample file download** — fetching test CRAM files from the htslib test suite
- **Full demo** — end-to-end parsing and display of results

In [ ]:
# Section 12: SAM output formatting

def _build_cigar(read_length: int, features: list, is_unmapped: bool) -> str:
    """Build a CIGAR string from read features."""
    if is_unmapped or not features:
        return '*' if is_unmapped else f'{read_length}M'

    ops = []
    read_pos = 1
    ref_consumed = 0

    for feat in sorted(features, key=lambda f: f['pos']):
        fc = feat['code']
        pos = feat['pos']

        # Add match bases before this feature
        if pos > read_pos:
            match_len = pos - read_pos
            ops.append((match_len, 'M'))
            read_pos = pos
            ref_consumed += match_len

        if fc == 'X':  # Substitution — still a match/mismatch in CIGAR
            ops.append((1, 'M'))
            read_pos += 1
            ref_consumed += 1
        elif fc == 'I':  # Insertion
            ins_len = len(feat.get('bases', b''))
            if ins_len > 0:
                ops.append((ins_len, 'I'))
                read_pos += ins_len
        elif fc == 'i':  # Single-base insertion
            ops.append((1, 'I'))
            read_pos += 1
        elif fc == 'D':  # Deletion
            del_len = feat.get('length', 1)
            ops.append((del_len, 'D'))
            ref_consumed += del_len
        elif fc == 'S':  # Soft clip
            sc_len = len(feat.get('bases', b''))
            if sc_len > 0:
                ops.append((sc_len, 'S'))
                read_pos += sc_len
        elif fc == 'H':  # Hard clip
            hc_len = feat.get('length', 1)
            ops.append((hc_len, 'H'))
        elif fc == 'N':  # Reference skip
            skip_len = feat.get('length', 1)
            ops.append((skip_len, 'N'))
            ref_consumed += skip_len
        elif fc == 'P':  # Padding
            pad_len = feat.get('length', 1)
            ops.append((pad_len, 'P'))
        elif fc == 'B':  # Read base (match/mismatch)
            ops.append((1, 'M'))
            read_pos += 1
            ref_consumed += 1
        elif fc == 'b':  # Bases
            b_len = len(feat.get('bases', b''))
            if b_len > 0:
                ops.append((b_len, 'M'))
                read_pos += b_len
                ref_consumed += b_len

    # Trailing matches
    if read_pos <= read_length:
        remaining = read_length - read_pos + 1
        if remaining > 0:
            ops.append((remaining, 'M'))

    if not ops:
        return f'{read_length}M'

    # Merge adjacent same-type operations
    merged = [ops[0]]
    for length, op in ops[1:]:
        if op == merged[-1][1]:
            merged[-1] = (merged[-1][0] + length, op)
        else:
            merged.append((length, op))

    return ''.join(f'{length}{op}' for length, op in merged)


def record_to_sam_line(record: dict, sam_header: Optional[dict] = None) -> str:
    """Convert a decoded CRAM record to a SAM text line."""
    if 'error' in record:
        return f"# Error: {record['error']}"

    qname = record.get('read_name', '*')
    flag = record.get('flags', 0)
    is_unmapped = bool(flag & BAM_FUNMAP)

    # Reference name
    rname = '*'
    ref_id = record.get('ref_id', -1)
    if ref_id >= 0 and sam_header and sam_header.get('SQ'):
        sq_list = sam_header['SQ']
        if ref_id < len(sq_list):
            rname = sq_list[ref_id].get('SN', '*')

    pos = record.get('align_start', 0) if not is_unmapped else 0
    mapq = record.get('mapping_quality', 0)
    cigar = _build_cigar(record.get('read_length', 0), record.get('features', []), is_unmapped)

    # Mate info
    rnext = '*'
    pnext = 0
    tlen = 0

    mate_ref_id = record.get('mate_ref_id', -1)
    if mate_ref_id >= 0 and sam_header and sam_header.get('SQ'):
        sq_list = sam_header['SQ']
        if mate_ref_id < len(sq_list):
            mate_rname = sq_list[mate_ref_id].get('SN', '*')
            if mate_rname == rname:
                rnext = '='
            else:
                rnext = mate_rname

    pnext = record.get('mate_align_start', 0)
    tlen = record.get('template_size', 0)

    seq = record.get('sequence', '*')
    quals = record.get('quality', [])
    if quals and any(q != 255 for q in quals):
        qual_str = ''.join(chr(q + 33) for q in quals)
    else:
        qual_str = '*'

    # Tags
    tag_strs = []
    for tag_name, tag_info in record.get('tags', {}).items():
        tag_type = tag_info.get('type', 'Z')
        tag_val = tag_info.get('value', '')
        if tag_type in ('i', 'c', 'C', 's', 'S', 'I'):
            tag_strs.append(f"{tag_name}:i:{tag_val}")
        elif tag_type == 'f':
            tag_strs.append(f"{tag_name}:f:{tag_val}")
        elif tag_type == 'A':
            tag_strs.append(f"{tag_name}:A:{tag_val}")
        elif tag_type == 'B':
            if isinstance(tag_val, list):
                arr_str = ','.join(str(v) for v in tag_val)
                tag_strs.append(f"{tag_name}:B:i,{arr_str}")
            else:
                tag_strs.append(f"{tag_name}:Z:{tag_val}")
        else:
            tag_strs.append(f"{tag_name}:Z:{tag_val}")

    fields = [qname, str(flag), rname, str(pos), str(mapq), cigar,
              rnext, str(pnext), str(tlen), seq, qual_str]
    if tag_strs:
        fields.extend(tag_strs)

    return '\t'.join(fields)

print("record_to_sam_line() and _build_cigar() defined.")

In [ ]:
# Section 12: Download sample CRAM files from htslib test suite

import urllib.request
import shutil

SAMPLE_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'cram_parser', 'samples')
os.makedirs(SAMPLE_DIR, exist_ok=True)

# htslib test files — small CRAM files suitable for testing
SAMPLE_URLS = {
    'ce#tag_padded.2.1.cram': 'https://raw.githubusercontent.com/samtools/htslib/develop/test/ce%23tag_padded.2.1.cram',
    'ce#tag_padded.2.1.cram.crai': 'https://raw.githubusercontent.com/samtools/htslib/develop/test/ce%23tag_padded.2.1.cram.crai',
    'c1#bounds.3.0.cram': 'https://raw.githubusercontent.com/samtools/htslib/develop/test/c1%23bounds.3.0.cram',
    'c1#noseq.3.0.cram': 'https://raw.githubusercontent.com/samtools/htslib/develop/test/c1%23noseq.3.0.cram',
}

def download_samples():
    """Download sample CRAM files for testing."""
    downloaded = []
    for filename, url in SAMPLE_URLS.items():
        filepath = os.path.join(SAMPLE_DIR, filename)
        if os.path.exists(filepath):
            print(f"  Already exists: {filename}")
            downloaded.append(filepath)
            continue
        try:
            print(f"  Downloading {filename}...")
            urllib.request.urlretrieve(url, filepath)
            size = os.path.getsize(filepath)
            print(f"    Saved ({size} bytes)")
            downloaded.append(filepath)
        except Exception as e:
            print(f"    Failed: {e}")
    return downloaded

print("Downloading sample CRAM files...")
sample_files = download_samples()
print(f"Available sample files: {len(sample_files)}")

In [ ]:
# Section 12: Full parsing demo

# Try parsing a CRAM v3.0 file
cram_v3_files = [f for f in sample_files if '3.0.cram' in f and not f.endswith('.crai')]

if cram_v3_files:
    cram_file = cram_v3_files[0]
    print(f"=== Parsing: {os.path.basename(cram_file)} ===\n")
    result = parse_cram(cram_file, max_records=50)

    # Display file header
    print("\n--- File Header ---")
    for k, v in result['file_header'].items():
        print(f"  {k}: {v}")

    # Display SAM header summary
    print("\n--- SAM Header ---")
    sh = result['sam_header']
    if sh['HD']:
        print(f"  @HD: {sh['HD']}")
    for sq in sh['SQ'][:5]:
        print(f"  @SQ: {sq}")
    if len(sh['SQ']) > 5:
        print(f"  ... and {len(sh['SQ']) - 5} more @SQ entries")
    for rg in sh['RG']:
        print(f"  @RG: {rg}")

    # Display container summaries
    print("\n--- Container Summaries ---")
    for cs in result['container_summaries']:
        print(f"  Container {cs['index']}: ref={cs['ref_seq_id']}, "
              f"pos={cs['start_pos']}, span={cs['align_span']}, "
              f"records={cs['num_records']}")

    # Display first records as SAM
    valid_records = [r for r in result['records'] if 'error' not in r]
    print(f"\n--- First 10 Records (SAM format) ---")
    for rec in valid_records[:10]:
        print(record_to_sam_line(rec, result['sam_header']))

    # Detailed view of first record
    if valid_records:
        print("\n--- Detailed First Record ---")
        rec = valid_records[0]
        print(f"  Read Name: {rec.get('read_name', '?')}")
        print(f"  Flags: {rec.get('flags', 0)} (0x{rec.get('flags', 0):04X})")
        print(f"  Ref ID: {rec.get('ref_id', -1)}")
        print(f"  Position: {rec.get('align_start', 0)}")
        print(f"  MAPQ: {rec.get('mapping_quality', 0)}")
        print(f"  Read Length: {rec.get('read_length', 0)}")
        print(f"  Sequence: {rec.get('sequence', '*')[:80]}...")
        print(f"  Features: {len(rec.get('features', []))}")
        for feat in rec.get('features', [])[:5]:
            print(f"    {feat}")
        print(f"  Tags: {list(rec.get('tags', {}).keys())}")
else:
    print("No CRAM v3.0 sample files available.")
    print("Trying any available CRAM file...")
    if sample_files:
        cram_file = sample_files[0]
        print(f"\nAttempting to parse: {os.path.basename(cram_file)}")
        try:
            result = parse_cram(cram_file, max_records=20)
            valid_records = [r for r in result['records'] if 'error' not in r]
            print(f"\nDecoded {len(valid_records)} records.")
            for rec in valid_records[:5]:
                print(record_to_sam_line(rec, result['sam_header']))
        except CRAMVersionError as e:
            print(f"Version error (expected for v2 files): {e}")
        except Exception as e:
            print(f"Error: {e}")

In [ ]:
# Section 12: CRAI Index demo

crai_files = [f for f in sample_files if f.endswith('.crai')]

if crai_files:
    crai_file = crai_files[0]
    print(f"=== CRAI Index: {os.path.basename(crai_file)} ===\n")

    try:
        entries = parse_crai(crai_file)
        print(f"Total index entries: {len(entries)}\n")

        print("Entries:")
        for entry in entries[:20]:
            print(f"  seq_id={entry['seq_id']}, start={entry['start']}, "
                  f"span={entry['span']}, container_offset={entry['container_offset']}, "
                  f"slice_offset={entry['slice_offset']}, slice_size={entry['slice_size']}")
        if len(entries) > 20:
            print(f"  ... and {len(entries) - 20} more entries")

        # Demo random access query
        if entries:
            first = entries[0]
            query_start = first['start']
            query_end = first['start'] + max(first['span'] // 2, 1)
            matches = random_access_query(entries, first['seq_id'], query_start, query_end)
            print(f"\nRandom access query: seq_id={first['seq_id']}, "
                  f"region={query_start}-{query_end}")
            print(f"  Matching entries: {len(matches)}")
            for m in matches:
                print(f"    offset={m['container_offset']}, "
                      f"start={m['start']}, span={m['span']}")
    except Exception as e:
        print(f"Error parsing CRAI: {e}")
else:
    print("No CRAI index files available.")